In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, quarter, year, month, to_date

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("SalesCalculation").getOrCreate()

# 2. Create Sample Data for sf_exchange_rate
exchange_data = [
    ("2020-01-01", 1.12, "EUR", "USD"),
    ("2020-01-15", 1.10, "EUR", "USD"),
    ("2020-02-20", 0.75, "GBP", "USD"), # Note: GBP usually > USD, just dummy data
    ("2020-04-05", 1.09, "EUR", "USD"),
    ("2020-05-12", 0.009, "JPY", "USD")
]
exchange_columns = ["date", "exchange_rate", "source_currency", "target_currency"]
df_exchange = spark.createDataFrame(exchange_data, exchange_columns)

# Convert string date to actual date type
df_exchange = df_exchange.withColumn("date", to_date(col("date")))


# 3. Create Sample Data for sf_sales_amount
sales_data = [
    ("EUR", 500, "2020-01-15"),  # Q1
    ("EUR", 300, "2020-01-01"),  # Q1
    ("GBP", 1000, "2020-02-20"), # Q1
    ("EUR", 400, "2020-04-05"),  # Q2
    ("JPY", 10000, "2020-05-12"),# Q2
    ("EUR", 600, "2021-01-01")   # Wrong Year (Should be filtered out)
]
sales_columns = ["currency", "sales_amount", "sales_date"]
df_sales = spark.createDataFrame(sales_data, sales_columns)

# Convert string date to actual date type
df_sales = df_sales.withColumn("sales_date", to_date(col("sales_date")))

# Create SQL Temp Views for the SQL part of the practice
df_exchange.createOrReplaceTempView("sf_exchange_rate")
df_sales.createOrReplaceTempView("sf_sales_amount")

print("Setup Complete. DataFrames created.")

In [0]:
df_exchange.display()
df_sales.display()

In [0]:
spark.sql("""
    SELECT
        QUARTER(s.sales_date) AS quarter,
        SUM(s.sales_amount * e.exchange_rate) AS total_sales_usd
    FROM sf_sales_amount s
    JOIN sf_exchange_rate e 
        ON s.currency = e.source_currency 
        AND e.target_currency = 'USD'
        AND e.date = s.sales_date
    WHERE YEAR(s.sales_date) = 2020
    AND QUARTER(s.sales_date) IN (1, 2)
    GROUP BY QUARTER(s.sales_date)
    ORDER BY quarter;
""").display()

In [0]:
df_joined = df_sales.alias('s').join(
    df_exchange.alias('e'), 
    (col('s.sales_date') == col('e.date')) & 
    (col('s.currency') == col('e.source_currency')),
    "inner"
)

In [0]:
df_joined.display()

In [0]:
result_df = df_joined \
    .filter(
        (col("target_currency") == "USD") & 
        (year(col("sales_date")) == 2020) & 
        (quarter(col("sales_date")).isin([1, 2])) 
    ) \
    .withColumn("total_sales_usd", col("sales_amount") * col("exchange_rate")) \
    .groupBy(quarter(col("sales_date")).alias("quarter")) \
    .agg(sum("total_sales_usd").alias("total_sales")) \
    .orderBy("quarter")